<a href="https://colab.research.google.com/github/AlexisSantiago21/CIIC5015-Project2-Neural-Networks/blob/main/clasificacion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto II: Redes Neuronales Fully Connected - Clasificación
**Estudiantes:** Alexis M. Santiago Rivera, Carlos S. Hernandez   
**Curso:** CIIC 5015 - Introducción a la IA  
**Objetivo:** Clasificar dígitos escritos a mano (0-9) del dataset MNIST usando PyTorch 2.0

Importaciones

In [ ]:
!pip install torch>=2.0
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Trabajando con: {device}")

PyTorch Version: 2.10.0+cu128
Trabajando con: cuda


Carga y Preparación de Datos

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.flatten(x))
])


train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)


train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)

print("Dataset MNIST cargado correctamente.")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.10MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.45MB/s]

Dataset MNIST cargado correctamente.


Definición de las 3 Redes

In [ ]:
model1 = nn.Sequential(
    nn.Linear(784, 20), nn.ReLU(),
    nn.Linear(20, 50), nn.ReLU(),
    nn.Linear(50, 20), nn.ReLU(),
    nn.Linear(20, 10), nn.LogSoftmax(dim=1)
).to(device)


model2 = nn.Sequential(
    nn.Linear(784, 10), nn.ReLU(),
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 30), nn.ReLU(),
    nn.Linear(30, 20), nn.ReLU(),
    nn.Linear(20, 10), nn.ReLU(),
    nn.Linear(10, 10), nn.LogSoftmax(dim=1)
).to(device)


model3 = nn.Sequential(
    nn.Linear(784, 10), nn.ReLU(),
    nn.Linear(10, 40), nn.ReLU(),
    nn.Linear(40, 70), nn.ReLU(),
    nn.Linear(70, 40), nn.ReLU(),
    nn.Linear(40, 10), nn.ReLU(),
    nn.Linear(10, 10), nn.LogSoftmax(dim=1)
).to(device)

Función de Entrenamiento y Tiempos

In [ ]:
def train_model(model):
    criterion = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    start_time = time.time()
    for epoch in range(20):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

    total_time = time.time() - start_time
    return total_time

print("Entrenando Modelo 1..."); t1 = train_model(model1)
print("Entrenando Modelo 2..."); t2 = train_model(model2)
print("Entrenando Modelo 3..."); t3 = train_model(model3)

Entrenando Modelo 1...
Entrenando Modelo 2...
Entrenando Modelo 3...


Evaluación y Respuesta

In [ ]:
def get_error(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 1 - (correct / total)

err1 = get_error(model1); err2 = get_error(model2); err3 = get_error(model3)

print(f"Error Validación - Mod 1: {err1:.4f} | Tiempo: {t1:.2f}s")
print(f"Error Validación - Mod 2: {err2:.4f} | Tiempo: {t2:.2f}s")
print(f"Error Validación - Mod 3: {err3:.4f} | Tiempo: {t3:.2f}s")

Error Validación - Mod 1: 0.0536 | Tiempo: 172.28s
Error Validación - Mod 2: 0.1319 | Tiempo: 179.87s
Error Validación - Mod 3: 0.0809 | Tiempo: 181.71s


Conclusión:
* Modelo con menor error: El Modelo #1 obtuvo el menor error de validación (0.0536). Esto se debe a que su capa de entrada de 20 neuronas permite una mejor extracción de características iniciales en comparación con el cuello de botella de 10 neuronas de los modelos de 6 capas.
* Tiempos de entrenamiento: El Modelo 1 fue el más rápido (172.28s), mientras que el Modelo 3 fue el más lento (181.71s) debido a su mayor cantidad de parámetros (capas de hasta 70 neuronas).